# LiCSBAS batch processing — python & shell

Two ways to run a full LiCSBAS processing chain with the **v3 layout**
(pip-installable `licsbas` package):

1. **Python batch** — `licsbas.batch`: a plain-Python config object
   (`BatchConfig`) that maps 1:1 to the old `batch_LiCSBAS.sh` settings,
   built on top of **predefined workflow templates** (`standard`, `reunw_5`,
   `reunw_10`, `reunw_20`, `sbovl`, `sbovl_abs`, ... — extendable with
   `@register_workflow`). Steps are executed *in-process* through
   `licsbas.steps.*`, and `save_flowchart()` renders a PNG flowchart of the
   chain (Graphviz `dot`; no extra python dependency).
2. **Shell batch** — the legacy `./batch_LiCSBAS.sh` invocation, which in v3 is
   a thin wrapper around `python -m licsbas.batch` (or the installed entry point
   `licsbas-batch`), so CLI flags like `--workflow reunw_5`, `--sbovl`,
   `--reunw`, `--check-only`, `--flowchart` all work.

All cells below run in **dry-run mode** by default (nothing is downloaded /
computed): set `DRY_RUN = False` in the first code cell — and have your LiCSAR
`GEOC` products in place — to process real data.


In [ ]:
import os, sys

# make the repo (or installed package) importable when running from a checkout
for _cand in (os.getcwd(), os.path.abspath(os.path.join(os.getcwd(), os.pardir))):
    if os.path.isdir(os.path.join(_cand, "licsbas")):
        sys.path.insert(0, _cand)
        break
import licsbas
print("using licsbas package:", licsbas.__file__)

# ---- toggle: True = only show what would be done (safe default) ----
DRY_RUN = True

## 1. Predefined workflow templates

Workflows are named `BatchConfig` factories — stable, reusable starting points.
`licsbas.batch` ships a small set (more can be added at any time); list them with
`list_workflows()` or `licsbas-batch --list-workflows`:


In [ ]:
from licsbas.batch import (BatchConfig, build_pipeline, run_batch,
                           save_flowchart, make_config, list_workflows,
                           register_workflow)

for name, desc in sorted(list_workflows().items()):
    print(f"{name:<12} {desc}")

### Standard chain (workflow `standard`)

Historical `batch_LiCSBAS.sh` defaults: 01 → 02 → 11 → 12 → 13 → 14 → 15 → 16.


In [ ]:
cfg = make_config("standard")   # == BatchConfig(start_step=1, end_step=16, nlook=1)
for s in build_pipeline(cfg):
    print(f"{s.key:>6}  {s.name:<34} {' '.join(s.argv)}")

png = save_flowchart(cfg, "flowchart_standard.png",
                     title="LiCSBAS standard processing chain")
try:
    from IPython.display import Image, display
    display(Image(filename=png))
except ImportError:
    pass  # outside jupyter: just open the file

### Re-unwrapping via `LiCSBAS02to05_unwrap.py` (workflows `reunw_5`, `reunw_10`, `reunw_20`)

The *updated* pipeline (`run_reunwrapping="y"` in the old script): step 01 also
downloads pha+ml; steps 02-05 are replaced by the single re-unwrapping script
(`--gacos`, `--hgtcorr`, `--cascade`, `--filter`, `--thres`, `-g`). The workflow
name encodes the multilook factor, e.g. **`reunw_5`** = 02to05 with `nlook=5`.


In [ ]:
cfg_reunw = make_config("reunw_5")                 # nlook=5, run_reunw=True
#cfg_reunw = make_config("reunw_10")               # or 10 / 20
cfg_reunw.p02to05_hgtcorr = True                   # recommended for high, varying terrain
#cfg_reunw.p02to05_gacos = True
#cfg_reunw.p02to05_filter = "gold"; cfg_reunw.p02to05_thres = "0.35"
#cfg_reunw.p02to05_cliparea_geo = "130.11/131.12/34.34/34.6"

png = save_flowchart(cfg_reunw, "flowchart_reunw_02to05.png",
                     title="LiCSBAS re-unwrapping chain via 02to05 (reunw_5)")
try:
    from IPython.display import Image, display
    display(Image(filename=png))
except ImportError:
    pass

### SBOI chain (workflows `sbovl`, `sbovl_abs`)

`sbovl=True` switches on the sbovl chain: `--sbovl` is passed to steps
01/02/11/13/15/16, step 120 (automated reference choice) is used, and step 12 is
skipped. `sbovl_abs=True` further closes the referencing and adds absolute
SET/Iono corrections via `LiCSBAS131_boi_absolute.py`.


In [ ]:
cfg_sbovl = make_config("sbovl")
#cfg_sbovl = make_config("sbovl_abs")              # absolute version
cfg_sbovl.nlook = 10
cfg_sbovl.p01_get_iono = True
cfg_sbovl.p01_get_tides = True
#cfg_sbovl.p131_tide = True                        # post-13 corrections
#cfg_sbovl.p131_iono = True                        # via the standard step
                                                 #   LiCSBAS131_corrections

png = save_flowchart(cfg_sbovl, "flowchart_sbovl.png",
                     title="LiCSBAS SBOI chain (--sbovl options)")
try:
    from IPython.display import Image, display
    display(Image(filename=png))
except ImportError:
    pass

## 2. Extending the workflow registry (your projects)

Templates are just named `BatchConfig` factories — register your own, e.g.:


In [ ]:
@register_workflow("my_study_2019",
                   "my standard SBOI study chain (nlook 10, 2014-2019)")
def _my_workflow():
    return BatchConfig(sbovl=True, nlook=10,
                       p01_start_date="20141001", p01_end_date="20191231")

cfg_mine = make_config("my_study_2019")
print(list_workflows()["my_study_2019"])
for s in build_pipeline(cfg_mine)[:3]:
    print(f"{s.key:>6}  {s.name:<34} {' '.join(s.argv)}")
# png = save_flowchart(cfg_mine, "flowchart_my_study.png")

## 3. Executing a chain

`run_batch(cfg, dry_run=True)` prints the commands (like the old
`check_only="y"`); `run_batch(cfg, dry_run=False)` executes the steps in-process
through `licsbas.steps.*` (a step-120 failure only warns and continues — same as
the old script).


In [ ]:
rc = run_batch(cfg, dry_run=DRY_RUN)
print("exit code:", rc)

## 4. Shell batch (legacy invocation)

`batch_LiCSBAS.sh` forwards to the same python batch, so the workflow templates
can be selected from the command line as well:

```bash
source bashrc_LiCSBAS.sh          # adds bin/ to PATH (legacy step wrappers)
./batch_LiCSBAS.sh --list-workflows
./batch_LiCSBAS.sh --workflow reunw_5 --check-only        # 02to05, multilook 5
./batch_LiCSBAS.sh --workflow standard --check-only       # old defaults
./batch_LiCSBAS.sh --workflow sbovl                        # --sbovl chain
./batch_LiCSBAS.sh --start 11 --end 16                     # continue products
./batch_LiCSBAS.sh --sbovl-abs --tide --iono               # flags override the template
./batch_LiCSBAS.sh --flowchart fc.png                      # also write the PNG flowchart
```

or, equivalently, as the installed console entry point:

```bash
licsbas-batch --list-workflows
licsbas-batch --workflow reunw_20 --check-only
```


## 5. Ready-made example scripts

| script | workflow | flowchart |
|--------|----------|-----------|
| `examples/example_1_standard.py` | `standard` | `flowchart_standard.png` |
| `examples/example_2_reunw_02to05.py` | re-unwrapping (02to05) | `flowchart_reunw_02to05.png` |
| `examples/example_3_sbovl.py` | `sbovl` | `flowchart_sbovl.png` |

```bash
python3 examples/example_1_standard.py            # dry run + PNG (default)
python3 examples/example_1_standard.py --execute  # really run it
```
